# Faz 5 — Yarışma Seti Üzerinde Harici Test

**Amaç:** Faz 3 (YOLOv8 2B) ve Faz 4 (nnDetection 3B) modellerini, eğitim sürecinde hiç görmediğimiz **Yarışma Veri Seti** üzerinde değerlendirip karşılaştırmak. Bu adım scoping review'da işaretlediğimiz "harici doğrulama eksikliği" boşluğunu doğrudan adresliyor.

**Çıktılar**
- Yarışma vakaları üzerinde her iki model için F1 @ IoU top-5 ortalama
- Sınıf bazında precision/recall/F1
- Hasta-düzeyi sensitivity/specificity (multi-label sınıflandırma kabuğu)
- Karşılaştırma tablosu (YOLO 2B vs nnDetection 3B)
- ROC/PR eğrileri (sınıf bazında)

**Süre tahmini:** YOLO inference ~30-60 dk, nnDetection inference 1-2 saat, değerlendirme <10 dk.

## 1. Ortam

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = DATA_ROOT / "Eğitim Verisi"
YARISMA_DIR = DATA_ROOT / "Yarışma Veri Seti"   # ZIP'i çıkardığınız klasör

os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
os.environ["ABDOMEN_TEST_DIR"]     = str(YARISMA_DIR)
os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
os.environ["ABDOMEN_OUT_DIR"]      = str(CODE / "outputs")

sys.path.insert(0, str(CODE))
from src.config import OUT_DIR, SUPER_CLASSES, RAW_PATHOLOGY_TO_SUPER, DEFAULT_WINDOWS

YARISMA_CASES = sorted([d for d in YARISMA_DIR.iterdir() if d.is_dir()]) if YARISMA_DIR.exists() else []
print(f"Yarışma vaka sayısı: {len(YARISMA_CASES)}")
print(f"İlk 5: {[d.name for d in YARISMA_CASES[:5]]}")

## 2. Yarışma Ground Truth (Bilgi.xlsx COMPETITIONDATA)

In [ ]:
import pandas as pd
import pydicom

comp_df = pd.read_excel(DATA_ROOT / "Bilgi.xlsx", sheet_name="COMPETITIONDATA")

# 6 üst sınıfa eşle
def to_super(cls):
    return RAW_PATHOLOGY_TO_SUPER.get(cls, -1)
comp_df['super_id'] = comp_df['Class'].map(to_super)

bbox_df = comp_df[(comp_df['Type'] == 'Bounding Box') & (comp_df['super_id'] >= 0)].copy()
print(f"Yarışma BBox annotasyonu: {len(bbox_df):,}")
print(f"Etkili vaka sayısı: {bbox_df['Case Number'].nunique()}")

# BBox parse
def parse_xywh(s):
    try:
        x1y1, x2y2 = s.split('-')
        return [int(v) for v in x1y1.split(',')] + [int(v) for v in x2y2.split(',')]
    except: return [None]*4

bbox_df[['x1','y1','x2','y2']] = bbox_df['Data'].apply(lambda s: pd.Series(parse_xywh(str(s))))
gt_df = bbox_df[['Case Number', 'Image Id', 'super_id', 'x1','y1','x2','y2']].rename(
    columns={'Case Number': 'case', 'Image Id': 'image_id', 'super_id': 'class'}
).dropna()
gt_df = gt_df.astype({'case':'int', 'image_id':'int', 'class':'int',
                      'x1':'int', 'y1':'int', 'x2':'int', 'y2':'int'})
print(f"GT satırı (temiz): {len(gt_df):,}")
gt_df.head()

## 3. YOLO 2B Modeli ile Yarışma Setinde Çıkarım

In [ ]:
from ultralytics import YOLO
import numpy as np
from src.dicom_utils import read_dicom, dicom_to_hu, hu_to_three_channel
from src.config import CKPT_DIR
from tqdm import tqdm

YOLO_WEIGHTS = CKPT_DIR / "yolo_fold0.pt"
print("YOLO ağırlık:", YOLO_WEIGHTS, "→ var?", YOLO_WEIGHTS.exists())

if YOLO_WEIGHTS.exists():
    model = YOLO(str(YOLO_WEIGHTS))
    CONF_TH = 0.05
    yolo_preds = []
    for case_dir in tqdm(YARISMA_CASES, desc="YOLO inference"):
        case = int(case_dir.name)
        for dcm in sorted(case_dir.glob("*.dcm")):
            img_id = int(dcm.stem)
            ds = read_dicom(dcm); hu = dicom_to_hu(ds)
            img = (hu_to_three_channel(hu, DEFAULT_WINDOWS) * 255).astype(np.uint8)
            res = model.predict(img, conf=CONF_TH, verbose=False, imgsz=512)[0]
            for box, score, cls in zip(res.boxes.xyxy.cpu().numpy(),
                                       res.boxes.conf.cpu().numpy(),
                                       res.boxes.cls.cpu().numpy()):
                yolo_preds.append({"case": case, "image_id": img_id, "class": int(cls),
                                   "score": float(score),
                                   "x1": float(box[0]), "y1": float(box[1]),
                                   "x2": float(box[2]), "y2": float(box[3])})
    yolo_pred_df = pd.DataFrame(yolo_preds)
    yolo_pred_df.to_csv(CODE/"outputs"/"logs"/"yolo_fold0_yarisma_preds.csv", index=False)
    print(f"YOLO yarışma tahminleri: {len(yolo_pred_df):,}")
else:
    print("YOLO ağırlığı yok — önce Faz 3'ü tamamlayın.")
    yolo_pred_df = pd.DataFrame()

## 4. nnDetection 3B Modeli ile Yarışma Setinde Çıkarım

In [ ]:
# nnDetection için yarışma vakaları NIfTI'ye dönüştürülmeli ve imagesTs altına konulmalı.
import SimpleITK as sitk
from concurrent.futures import ThreadPoolExecutor
import subprocess, shutil

NND_ROOT = OUT_DIR / "nndet"
TASK_DIR = NND_ROOT / "raw" / "Task100_Abdomen" / "raw_splitted"
TEST_NII_DIR = NND_ROOT / "nifti_test"
TEST_NII_DIR.mkdir(parents=True, exist_ok=True)
(TASK_DIR / "imagesTs").mkdir(parents=True, exist_ok=True)

def dcm_dir_to_nifti(case_dir: Path, out_path: Path):
    if out_path.exists(): return "skip"
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names: return "no_dcm"
    reader.SetFileNames(names)
    try:
        img = reader.Execute()
        sitk.WriteImage(img, str(out_path))
        return "ok"
    except Exception as e:
        return f"err:{e}"

def _conv(case_dir):
    case = int(case_dir.name)
    out_nii = TEST_NII_DIR / f"case_{case:05d}_0000.nii.gz"
    s = dcm_dir_to_nifti(case_dir, out_nii)
    # imagesTs'e bağla
    target = TASK_DIR / "imagesTs" / f"case_{case:05d}_0000.nii.gz"
    if not target.exists() and out_nii.exists():
        try: os.symlink(out_nii, target)
        except (OSError, NotImplementedError): shutil.copy2(out_nii, target)
    return case, s

with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(tqdm(ex.map(_conv, YARISMA_CASES), total=len(YARISMA_CASES),
                        desc="Yarışma DICOM→NIfTI"))
print(f"\n✓ {sum(1 for _,s in results if s=='ok')} yeni, {sum(1 for _,s in results if s=='skip')} atlandı")

In [ ]:
# nnDetection ile yarışma seti üzerinde inference
os.environ["det_data"]    = str(NND_ROOT / "raw")
os.environ["det_models"]  = str(NND_ROOT / "models")
print("nnDetection predict (test set)...")
try:
    r = subprocess.run(["nndet_predict", "100", "RetinaUNetV001_D3V001_3d",
                        "--fold", "0", "--test", "True"],
                       capture_output=True, text=True, env=os.environ, timeout=3600*4)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-1000:])
except FileNotFoundError:
    print("⚠️  nndet_predict bulunamadı; manuel: nndet_predict 100 RetinaUNetV001_D3V001_3d --fold 0 --test True")
except subprocess.TimeoutExpired:
    print("Süre aşıldı; arka planda devam ediyor olabilir.")

In [ ]:
# 3B → 2D projeksiyon (Faz 4'teki yöntem)
TEST_PRED_DIR = NND_ROOT / "models" / "Task100_Abdomen" / "RetinaUNetV001_D3V001_3d" / "fold0" / "test_predictions"

def load_3d_preds_as_2d(pred_dir: Path) -> pd.DataFrame:
    rows = []
    if not pred_dir.exists(): return pd.DataFrame(rows)
    for pkl in pred_dir.glob("*_boxes.json"):
        case = int(pkl.stem.split("_")[1])
        data = json.loads(pkl.read_text())
        for b, s, c in zip(data.get("boxes",[]), data.get("scores",[]), data.get("classes",[])):
            x1,y1,z1,x2,y2,z2 = b
            for z in range(int(z1), int(z2)+1):
                rows.append({"case": case, "image_id": z, "class": int(c),
                             "score": float(s),
                             "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return pd.DataFrame(rows)

nnd_pred_df = load_3d_preds_as_2d(TEST_PRED_DIR)
print(f"nnDetection yarışma tahminleri (2D-proj): {len(nnd_pred_df):,}")
if not nnd_pred_df.empty:
    nnd_pred_df.to_csv(CODE/"outputs"/"logs"/"nndet_fold0_yarisma_preds.csv", index=False)

## 5. Karşılaştırmalı Değerlendirme

In [ ]:
from src.evaluation import top5_f1_mean, f1_at_iou

def evaluate(name, pred_df, gt_df):
    if pred_df.empty:
        return {"model": name, "n_pred": 0, "top5_f1": None}
    top5 = top5_f1_mean(pred_df, gt_df)
    detail03 = f1_at_iou(pred_df, gt_df, 0.3)
    detail05 = f1_at_iou(pred_df, gt_df, 0.5)
    return {
        "model": name,
        "n_pred": len(pred_df),
        "top5_f1": round(top5['top5_mean_f1'], 4),
        "macro_f1@0.3": round(detail03['macro_f1'], 4),
        "macro_f1@0.5": round(detail05['macro_f1'], 4),
        "micro_f1@0.3": round(detail03['micro_f1'], 4),
    }

comp = pd.DataFrame([
    evaluate("YOLOv8 2B (Faz 3)", yolo_pred_df, gt_df),
    evaluate("nnDetection 3B (Faz 4)", nnd_pred_df, gt_df),
])
comp

In [ ]:
# Sınıf bazında karşılaştırma — IoU 0.3
rows = []
for model_name, pred_df in [("YOLO_2B", yolo_pred_df), ("nnDet_3B", nnd_pred_df)]:
    if pred_df.empty: continue
    d = f1_at_iou(pred_df, gt_df, 0.3)
    for cid, m in d['per_class'].items():
        if cid >= len(SUPER_CLASSES): continue
        rows.append({
            "model": model_name,
            "class": SUPER_CLASSES[cid],
            "precision": round(m['precision'], 3),
            "recall": round(m['recall'], 3),
            "f1": round(m['f1'], 3),
            "TP": m['tp'], "FP": m['fp'], "FN": m['fn'],
        })
class_table = pd.DataFrame(rows).pivot_table(index='class', columns='model',
    values=['precision','recall','f1'], aggfunc='first')
class_table

## 6. Hasta-Düzeyi Multi-Label Değerlendirme

In [ ]:
# Her vaka için: model en az 1 BBox tahmin etmişse o vakada patolojinin "var" olduğu söylenir
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np

def patient_level(pred_df, gt_df):
    """Vaka × sınıf matrisleri (binary)."""
    all_cases = sorted(set(pred_df['case']).union(set(gt_df['case'])))
    n_cls = len(SUPER_CLASSES)
    y_true = np.zeros((len(all_cases), n_cls), dtype=int)
    y_pred = np.zeros((len(all_cases), n_cls), dtype=int)
    y_score = np.zeros((len(all_cases), n_cls), dtype=float)
    idx = {c: i for i, c in enumerate(all_cases)}
    for _, r in gt_df.iterrows():
        i = idx.get(int(r['case']))
        if i is None or int(r['class']) >= n_cls: continue
        y_true[i, int(r['class'])] = 1
    if not pred_df.empty:
        for case, grp in pred_df.groupby('case'):
            i = idx.get(int(case))
            if i is None: continue
            for cls, sub in grp.groupby('class'):
                if int(cls) >= n_cls: continue
                y_pred[i, int(cls)] = 1
                y_score[i, int(cls)] = float(sub['score'].max())
    return y_true, y_pred, y_score, all_cases

for model_name, pred_df in [("YOLO_2B", yolo_pred_df), ("nnDet_3B", nnd_pred_df)]:
    if pred_df.empty: continue
    yt, yp, ys, _ = patient_level(pred_df, gt_df)
    print(f"\n=== {model_name} — Hasta düzeyi ===")
    for cid in range(len(SUPER_CLASSES)):
        if yt[:, cid].sum() < 2: continue
        try:
            auc = roc_auc_score(yt[:, cid], ys[:, cid])
        except: auc = float('nan')
        sens = yp[yt[:, cid]==1, cid].mean() if yt[:, cid].sum() else 0
        spec = 1 - yp[yt[:, cid]==0, cid].mean() if (yt[:, cid]==0).sum() else 0
        print(f"  {SUPER_CLASSES[cid]:30s} AUC={auc:.3f}  Sens={sens:.3f}  Spec={spec:.3f}  (n_pos={int(yt[:,cid].sum())})")

## 7. Sonuçları Diske Kaydet

In [ ]:
out = CODE / "outputs" / "logs" / "yarisma_karsilastirma.json"
result = {
    "summary": comp.to_dict(orient='records'),
    "n_yarisma_cases": len(YARISMA_CASES),
    "n_yarisma_gt_bboxes": len(gt_df),
}
out.write_text(json.dumps(result, indent=2, ensure_ascii=False))
print("Sonuç kaydı:", out)
print("\n📊 ÖZET:")
print(comp.to_string(index=False))

## 8. Faz 5 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| YOLO yarışma tahminleri | `outputs/logs/yolo_fold0_yarisma_preds.csv` |
| nnDetection yarışma tahminleri | `outputs/logs/nndet_fold0_yarisma_preds.csv` |
| Karşılaştırma raporu | `outputs/logs/yarisma_karsilastirma.json` |

**Makale için kritik tablo:** YOLO 2B vs nnDetection 3B (top-5 F1, macro F1@0.3, macro F1@0.5) + hasta-düzeyi AUC/sens/spec.

**Sonraki adım:** Faz 6 → ablation çalışmaları (HU pencereleri, model boyutu, post-processing on/off).